# Phase 8: Proposed Model (ASTGCNLite)

In [1]:
import sys
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

if "google.colab" in sys.modules:
    if "/content" not in sys.path:
        sys.path.insert(0, "/content")
else:
    sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

from project_paths import *

In [2]:
def load_npz(path):
    data = np.load(path)
    return data['X'], data['Y']

x_train, y_train = load_npz(PROCESSED_DATA_DIR / "train" / "sequences.npz")
x_val, y_val = load_npz(PROCESSED_DATA_DIR / "validation" / "sequences.npz")
x_test, y_test = load_npz(PROCESSED_DATA_DIR / "test" / "sequences.npz")

print(f"Train shape: {x_train.shape} -> {y_train.shape}")
print(f"Val shape: {x_val.shape} -> {y_val.shape}")
print(f"Test shape: {x_test.shape} -> {y_test.shape}")

history_steps = x_test.shape[1]
horizon_steps = y_test.shape[1]
nodes = x_test.shape[2]
features = x_test.shape[3]

print(f"\nExtracted Config:")
print(f"History: {history_steps}")
print(f"Horizon: {horizon_steps}")
print(f"Nodes: {nodes}")
print(f"Features: {features}")

Train shape: (23976, 12, 207, 1) -> (23976, 3, 207, 1)
Val shape: (5126, 12, 207, 1) -> (5126, 3, 207, 1)
Test shape: (5128, 12, 207, 1) -> (5128, 3, 207, 1)

Extracted Config:
History: 12
Horizon: 3
Nodes: 207
Features: 1


In [3]:
from src.data.dataset import get_dataloaders

batch_size = 64
train_loader, val_loader, test_loader = get_dataloaders(
    x_train, y_train,
    x_val, y_val,
    x_test, y_test,
    batch_size=batch_size
)

batch_x, batch_y = next(iter(test_loader))
print(f"Batch X: {batch_x.shape}")
print(f"Batch Y: {batch_y.shape}")

Batch X: torch.Size([64, 12, 207, 1])
Batch Y: torch.Size([64, 3, 207, 1])


In [4]:
from src.models.proposed import ASTGCNLite
from src.training.train import train_model
from src.evaluation.evaluate import evaluate_model_comprehensive
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
blocks = [[16, 64, 3], [64, 64, 3]]
model = ASTGCNLite(blocks, n_nodes=nodes, history_steps=history_steps, horizon_steps=horizon_steps, features=features)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.L1Loss()

In [5]:
print("Training ASTGCNLite...")
train_model(model, train_loader, val_loader, optimizer, loss_fn, num_epochs=3, device=device)

Training ASTGCNLite...


Epoch 1/3 - Train Loss: 0.3850 - Val Loss: 0.3491 - Val MAE: 0.3491


Epoch 2/3 - Train Loss: 0.3128 - Val Loss: 0.3193 - Val MAE: 0.3193


Epoch 3/3 - Train Loss: 0.2919 - Val Loss: 0.2961 - Val MAE: 0.2961


{'train_loss': [0.3850473388656601, 0.312828455879802, 0.29188835744186364],
 'val_loss': [0.3491036914262835, 0.3193396792147153, 0.29607460050437767],
 'val_mae': [0.3491036914262835, 0.3193396792147153, 0.29607460050437767]}

In [6]:
results = evaluate_model_comprehensive(model, test_loader, device)
print("ASTGCNLite Metrics:", results['overall'])
model_path = PROJECT_ROOT / 'experiments' / 'astgcnlite_weights.pt'
torch.save(model.state_dict(), model_path)
print(f"Saved weights to {model_path}")

ASTGCNLite Metrics: {'MAE': 0.31058835983276367, 'RMSE': 0.6803553104400635, 'MAPE': 0.08257240802049637, 'WAPE': 0.37522682547569275, 'R2': 0.6830300092697144}
Saved weights to /Users/sushank/Documents/GitHub/urbanAi/experiments/astgcnlite_weights.pt
